# Statistical Methods in Imaging (SMI) Conference, 2026.
# Empowering Large Language Models with Statistics: A Practical Tutorial for Medical Imaging
**Ernest (Khashayar) Namdar, Dominik A. Deniffel, Pascal Tyrrell**

This tutorial focuses on classifying Acute Ischemic Stroke (AIS) from Radiology reports. We will use the `Label` and `Text` columns for a binary classification task.

Several similar pipelines were discussed in our publication:
```bibtex
@inproceedings{10.1117/12.3084682,
author = {Khashayar Namdar and Saeidehsadat Mirjalili and Lauren Erdman and Dominik A. Deniffel and Keith Brunt and Leo Anthony Celi},
title = {{Comparative evaluation of machine learning and large language model pipelines for identifying acute ischemic stroke in radiology reports}},
volume = {13926},
booktitle = {Medical Imaging 2026: Computer-Aided Diagnosis},
editor = {Axel Wism{\"u}ller and Thomas Martin Deserno},
organization = {International Society for Optics and Photonics},
publisher = {SPIE},
pages = {139261S},
keywords = {Stroke, NLP, Machine Learning, Large Language Models},
year = {2026},
doi = {10.1117/12.3084682},
URL = {https://doi.org/10.1117/12.3084682}
}
```

Also, the source for the dataset is:
```bibtex
@article{10.1371/journal.pone.0212778,
    doi = {10.1371/journal.pone.0212778},
    author = {Kim, Chulho AND Zhu, Vivienne AND Obeid, Jihad AND Lenert, Leslie},
    journal = {PLOS ONE},
    publisher = {Public Library of Science},
    title = {Natural language processing and machine learning algorithm to identify brain MRI reports with acute ischemic stroke},
    year = {2019},
    month = {02},
    volume = {14},
    url = {https://doi.org/10.1371/journal.pone.0212778},
    pages = {1-13},
    number = {2},
}
```


## Model Agreement Analysis
We will read the predictions from all three models (LightGBM on Embeddings, LightGBM on TF-IDF, and Random Forest on Embeddings) and evaluate the performance of an ensemble average based on whether the models completely agreed or partially disagreed on the class label (threshold = 0.5).

In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
import scipy.stats

# Load data
df = pd.read_csv('../data/AIS.csv')

# Load the three prediction files
# Each has: ID, predictions, Label
pred_lgb = pd.read_csv('../results/predictions.csv')
pred_tfidf = pd.read_csv('../results/predictions_tfidf.csv')
pred_rf = pd.read_csv('../results/predictions_rf.csv')

# Merge them all on ID (and Label to ensure consistency)
data = pred_lgb.rename(columns={'predictions': 'pred_lgb'})
data = data.merge(pred_tfidf[['ID', 'predictions']].rename(columns={'predictions': 'pred_tfidf'}), on='ID')
data = data.merge(pred_rf[['ID', 'predictions']].rename(columns={'predictions': 'pred_rf'}), on='ID')

# Convert probabilities to hard classes (threshold = 0.5)
data['class_lgb'] = (data['pred_lgb'] >= 0.5).astype(int)
data['class_tfidf'] = (data['pred_tfidf'] >= 0.5).astype(int)
data['class_rf'] = (data['pred_rf'] >= 0.5).astype(int)

# Sum the predicted classes (0 to 3)
data['agree_sum'] = data['class_lgb'] + data['class_tfidf'] + data['class_rf']

# Scenario 1: All 3 models agree (sum is 0 or 3)
mask_all_agree = (data['agree_sum'] == 0) | (data['agree_sum'] == 3)

# Scenario 2: One model disagrees (sum is 1 or 2)
mask_one_disagrees = (data['agree_sum'] == 1) | (data['agree_sum'] == 2)

# Average probability for ensemble AUC
data['ensemble_prob'] = (data['pred_lgb'] + data['pred_tfidf'] + data['pred_rf']) / 3.0

def report_metrics(subset_data, scenario_name):
    y_true = subset_data['Label'].values
    y_scores = subset_data['ensemble_prob'].values
    n_samples = len(y_true)
    
    print(f"\n{'='*50}")
    print(f"Scenario: {scenario_name}")
    print(f"N (samples): {n_samples}")
    print(f"{'='*50}")
    
    if len(np.unique(y_true)) > 1:
        auc = roc_auc_score(y_true, y_scores)
        
        # Calculate AUC variance and CI using Hanley-McNeil or bootstrap
        # Since we just want a CI and scipy.stats on folds isn't applicable for the whole subset here,
        # we will use a bootstrap approach to calculate the 95% CI of the AUC
        n_bootstraps = 1000
        bootstrapped_scores = []
        rng = np.random.RandomState(42)
        for i in range(n_bootstraps):
            # bootstrap by sampling with replacement on the indices
            indices = rng.randint(0, len(y_scores), len(y_scores))
            if len(np.unique(y_true[indices])) < 2:
                # We need at least one positive and one negative sample for ROC AUC
                continue
            score = roc_auc_score(y_true[indices], y_scores[indices])
            bootstrapped_scores.append(score)
            
        sorted_scores = np.array(bootstrapped_scores)
        sorted_scores.sort()
        ci_lower = sorted_scores[int(0.025 * len(sorted_scores))]
        ci_upper = sorted_scores[int(0.975 * len(sorted_scores))]
        
        print(f"Ensemble AUC: {auc:.4f}")
        print(f"95% CI: [{ci_lower:.4f}, {ci_upper:.4f}]")
    else:
        print("Not enough class diversity to calculate AUC (only one class present in this subset).")

report_metrics(data[mask_all_agree], "All 3 models agreed")
report_metrics(data[mask_one_disagrees], "1 model did not agree (Majority Vote)")



Scenario: All 3 models agreed
N (samples): 2748
Ensemble AUC: 0.9935
95% CI: [0.9863, 0.9982]

Scenario: 1 model did not agree (Majority Vote)
N (samples): 276
Ensemble AUC: 0.8321
95% CI: [0.7822, 0.8812]
